# Import Libraries

In [ ]:
import os
import sys
import glob
import json
import shutil
import re
import random
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, GlobalAveragePooling2D,
    Dense, Dropout, Flatten, Concatenate, BatchNormalization,
    TimeDistributed, LSTM, Bidirectional
)
from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import cv2
import yaml
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split


In [ ]:
# Install required packages if not already installed
try:
    import cv2
except ImportError:
    !pip install opencv-python-headless

try:
    from tqdm import tqdm
except ImportError:
    !pip install tqdm

# Part 1: Configuration and Setup

In [ ]:
# Project paths setup
project_root = '/content/drive/MyDrive/Colab Notebooks/Pawnder'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to import from config
try:
    from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
except ModuleNotFoundError:
    print("Could not import config module. Adjusting path...")
    # Try with full path
    sys.path.insert(0, os.path.abspath(project_root))
    try:
        from config import DATA_DIRS, ensure_directories, EMOTION_MAPPING
    except ModuleNotFoundError:
        # Define default configuration if import fails
        print("Creating default configuration")
        
        def normalize_path(path):
            """Normalize a path to avoid duplicates caused by different representations"""
            if isinstance(path, str) and path.startswith('s3://'):
                return path  # Don't normalize S3 paths
            return os.path.normpath(path)
        
        # Default data directories
        base_dir = "/content/drive/MyDrive/Colab Notebooks/Pawnder"
        DATA_ROOT = normalize_path(os.path.join(base_dir, 'Data'))
        
        DATA_DIRS = {
            'raw': os.path.join(DATA_ROOT, 'Raw'),
            'stanford_original': os.path.join(DATA_ROOT, 'Raw', 'stanford_dog_pose'),
            'personal_dataset': os.path.join(DATA_ROOT, 'Raw', 'personal_dataset'),
            'matrix': os.path.join(DATA_ROOT, 'Matrix'),
            'interim': os.path.join(DATA_ROOT, 'interim'),
            'processed': os.path.join(DATA_ROOT, 'processed'),
            'models': os.path.join(base_dir, 'Models'),
        }
        
        # Default emotion mapping
        EMOTION_MAPPING = {
            "Happy or Playful": "Happy/Playful",
            "Relaxed": "Relaxed",
            "Submissive": "Submissive/Appeasement",
            "Excited": "Happy/Playful",
            "Drowsy or Bored": "Relaxed",
            "Curious or Confused": "Curiosity/Alertness",
            "Confident or Alert": "Curiosity/Alertness",
            "Jealous": "Stressed",
            "Stressed": "Stressed",
            "Frustrated": "Stressed",
            "Unsure or Uneasy": "Fearful/Anxious",
            "Possessive, Territorial, Dominant": "Aggressive/Threatening",
            "Fear or Aggression": "Fearful/Anxious",
            "Pain": "Stressed"
        }
        
        def ensure_directories():
            """Ensure all required directories exist"""
            for path in DATA_DIRS.values():
                if isinstance(path, str) and not path.startswith('s3://'):
                    Path(path).mkdir(parents=True, exist_ok=True)
            
            # Create subdirectories for processed data
            for split in ['train', 'validation', 'test']:
                split_path = os.path.join(DATA_DIRS['processed'], split)
                for subdir in ['images', 'annotations']:
                    path = os.path.join(split_path, subdir)
                    Path(path).mkdir(parents=True, exist_ok=True)

# Make sure directories exist
ensure_directories()

# Define configuration for the model (can be loaded from YAML if available)
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file or use defaults"""
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        return config
    else:
        # Use default configuration
        print(f"Config file not found: {config_path}, using defaults")
        default_config = {
            "data": {
                "base_dir": project_root,
                "raw_data_dir": os.path.join(DATA_DIRS['raw']),
                "processed_data_dir": os.path.join(DATA_DIRS['processed']),
                "train_split": 0.7,
                "val_split": 0.15,
                "test_split": 0.15
            },
            "model": {
                "image_size": [224, 224, 3],
                "batch_size": 32,
                "learning_rate": 0.001,
                "dropout_rate": 0.5,
                "backbone": "mobilenetv2",
                "early_stopping_patience": 5
            },
            "training": {
                "checkpoint_dir": os.path.join(DATA_DIRS['models'], "checkpoints"),
                "logs_dir": os.path.join(DATA_DIRS['models'], "logs")
            },
            "inference": {
                "confidence_threshold": 0.6,
                "behavior_threshold": 0.5,
                "output_dir": os.path.join(DATA_DIRS['processed'], "predictions")
            }
        }
        
        # Save default config for future use
        os.makedirs(os.path.dirname(config_path), exist_ok=True)
        with open(config_path, 'w') as f:
            yaml.dump(default_config, f)
            
        return default_config

# Load configuration
config = load_config()

# Define emotion classes from the EMOTION_MAPPING
CLASS_NAMES = list(set(EMOTION_MAPPING.values()))
# Safe class names for directories
SAFE_CLASS_NAMES = [emotion.replace("/", "_").replace("\\", "_") for emotion in CLASS_NAMES]
CLASS_MAP = dict(zip(CLASS_NAMES, SAFE_CLASS_NAMES))


# PART 2: Data Processing Pipeline

In [ ]:
def parse_xml_annotation(xml_file):
    """Parse CVAT XML annotations"""
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Get video name from task source
        video_name = None
        for meta in root.findall('.//meta'):
            for task in meta.findall('.//task'):
                for source in task.findall('.//source'):
                    video_name = source.text
                    if video_name:
                        # Remove extension if present
                        video_name = os.path.splitext(video_name)[0]

        if not video_name:
            video_name = os.path.basename(os.path.dirname(xml_file))

        # Extract annotations
        annotations = {}

        for track in root.findall('.//track'):
            for box in track.findall('.//box'):
                frame_num = int(box.get('frame', 0))
                frame_id = f"frame_{frame_num:06d}"

                # Look for Primary Emotion attribute
                emotion = None
                for attr in box.findall('.//attribute'):
                    name = attr.get('name')
                    if name == "Primary Emotion":
                        emotion = attr.text
                        break

                if emotion:
                    # Map to standardized emotion if needed
                    if emotion in EMOTION_MAPPING:
                        emotion = EMOTION_MAPPING[emotion]

                    annotations[frame_id] = {
                        "emotions": {"primary_emotion": emotion},
                        "video_name": video_name,
                        "frame": frame_num,
                        "original_format": "xml"
                    }

        return annotations, video_name

    except Exception as e:
        print(f"Error parsing XML file {xml_file}: {str(e)}")
        return {}, None

def process_video_folder(video_folder, combined_frames_dir, output_dir):
    """Process a video folder with images and annotations"""
    folder_name = os.path.basename(video_folder)
    print(f"\nProcessing video folder: {folder_name}")

    # Find any XML file in this folder
    xml_files = glob.glob(os.path.join(video_folder, "*.xml"))
    if not xml_files:
        print(f"  No XML files found in {video_folder}")
        return 0, {}

    # Use the first XML file
    xml_file = xml_files[0]
    print(f"  Using XML file: {os.path.basename(xml_file)}")

    # Parse annotations
    annotations, video_name = parse_xml_annotation(xml_file)
    if not video_name:
        video_name = folder_name

    # Generate video ID
    video_id = ''.join(e for e in video_name if e.isalnum())[:8]

    # Look for images directory
    images_dir = os.path.join(video_folder, "images")
    if not os.path.exists(images_dir):
        print(f"  Images directory not found: {images_dir}")
        return 0, {}

    # Count annotations by emotion
    emotion_counts = defaultdict(int)
    for frame_id, data in annotations.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print(f"  Found {len(annotations)} annotated frames")
    print("  Emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    # Get all files in the images directory
    all_files = os.listdir(images_dir)
    print(f"  Found {len(all_files)} files in images directory")

    # Create filename mapping for quick lookup
    filename_map = {}
    for filename in all_files:
        match = re.search(r'frame_0*(\d+)', filename.lower())
        if match:
            frame_num = int(match.group(1))
            if frame_num not in filename_map:
                filename_map[frame_num] = []
            filename_map[frame_num].append(filename)

    # Process each file
    processed_frames = {}
    processed_count = 0
    missing_count = 0

    for frame_id, annotation in tqdm(annotations.items(), desc=f"Processing {folder_name}", leave=False):
        # Extract frame number
        match = re.search(r'frame_0*(\d+)', frame_id)
        if not match:
            continue

        frame_num = int(match.group(1))

        # Find matching file
        if frame_num in filename_map and filename_map[frame_num]:
            src_filename = filename_map[frame_num][0]
            src_path = os.path.join(images_dir, src_filename)

            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename with consistent format
            new_filename = f"video_{video_id}_frame_{frame_num:06d}{os.path.splitext(src_filename)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)

            # Copy to class directory
            class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
            class_path = os.path.join(class_dir, new_filename)
            os.makedirs(os.path.dirname(class_path), exist_ok=True)
            shutil.copy2(src_path, class_path)

            # Add to processed frames
            processed_frames[new_filename] = {
                "emotions": {"primary_emotion": emotion},
                "video_name": video_name,
                "video_id": video_id,
                "frame_id": frame_id,
                "original_path": src_path,
                "source": "video_frames"
            }

            processed_count += 1
        else:
            missing_count += 1

    print(f"  Processed {processed_count} frames, {missing_count} frames missing")
    return processed_count, processed_frames

def process_personal_images(personal_images_dir, emotions_file, combined_frames_dir, output_dir):
    """Process personal dataset images"""
    print("\nProcessing Personal Dataset Images")

    # Check if emotions_only.json exists
    if not os.path.exists(emotions_file):
        print(f"  Error: emotions_only.json not found at {emotions_file}")
        return 0, {}

    # Load the emotions file
    try:
        with open(emotions_file, 'r') as f:
            all_annotations = json.load(f)
        print(f"  Loaded {len(all_annotations)} annotations from emotions_only.json")
    except Exception as e:
        print(f"  Error loading emotions file: {str(e)}")
        return 0, {}

    # Check if personal_images_dir exists
    if not os.path.exists(personal_images_dir):
        print(f"  Error: Personal images directory not found at {personal_images_dir}")
        return 0, {}

    # Build a map of all personal images
    personal_images = {}
    image_files = []

    # Look for images in the root and subdirectories
    for root, dirs, files in os.walk(personal_images_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                file_path = os.path.join(root, file)
                image_files.append(file_path)
                # Add with basename as key
                personal_images[file] = file_path
                # Also add without extension
                name_without_ext = os.path.splitext(file)[0]
                personal_images[name_without_ext] = file_path

    print(f"  Found {len(image_files)} total personal images")

    # Find personal images in annotations
    processed_frames = {}
    processed_count = 0

    for img_id, annotation in tqdm(all_annotations.items(), desc="Processing personal annotations"):
        # Skip if this doesn't have an emotion
        if "emotions" not in annotation or "primary_emotion" not in annotation["emotions"]:
            continue

        # Get the filename
        filename = os.path.basename(img_id)
        basename = os.path.splitext(filename)[0]

        # Check if we have this image
        image_path = None
        if filename in personal_images:
            image_path = personal_images[filename]
        elif basename in personal_images:
            image_path = personal_images[basename]

        # Skip if it's a Stanford image
        is_stanford = re.match(r'n\d+_\d+', basename) is not None
        if is_stanford:
            continue

        # Skip if it's a video frame (already processed)
        is_video_frame = "frame_" in basename
        if is_video_frame:
            continue

        if image_path:
            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename
            new_filename = f"personal_{basename}{os.path.splitext(image_path)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            try:
                shutil.copy2(image_path, dst_path)

                # Copy to class directory
                class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
                class_path = os.path.join(class_dir, new_filename)
                os.makedirs(os.path.dirname(class_path), exist_ok=True)
                shutil.copy2(image_path, class_path)

                # Add to processed frames
                processed_frames[new_filename] = {
                    "emotions": {"primary_emotion": emotion},
                    "original_id": img_id,
                    "original_path": image_path,
                    "source": "personal"
                }

                processed_count += 1

                if processed_count % 100 == 0:
                    print(f"  Processed {processed_count} personal images")

            except Exception as e:
                print(f"  Error copying {image_path}: {str(e)}")

    print(f"  Processed {processed_count} personal images with emotion annotations")

    # Count by emotion
    emotion_counts = defaultdict(int)
    for _, data in processed_frames.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print("  Personal dataset emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    return processed_count, processed_frames

def process_stanford_dataset(stanford_images_dir, emotions_file, combined_frames_dir, output_dir):
    """Process Stanford dog dataset images"""
    print("\nProcessing Stanford Dog Dataset")

    # Check if emotions_only.json exists
    if not os.path.exists(emotions_file):
        print(f"  Error: emotions_only.json not found at {emotions_file}")
        return 0, {}

    # Load the emotions file
    try:
        with open(emotions_file, 'r') as f:
            all_annotations = json.load(f)
        print(f"  Loaded {len(all_annotations)} annotations from emotions_only.json")
    except Exception as e:
        print(f"  Error loading emotions file: {str(e)}")
        return 0, {}

    # Check if stanford_images_dir exists
    if not os.path.exists(stanford_images_dir):
        print(f"  Error: Stanford images directory not found at {stanford_images_dir}")
        return 0, {}

    # Find all stanford breed directories
    breed_dirs = []
    for item in os.listdir(stanford_images_dir):
        breed_path = os.path.join(stanford_images_dir, item)
        if os.path.isdir(breed_path) and item.startswith('n'):  # Stanford uses n* format for breed directories
            breed_dirs.append(breed_path)

    print(f"  Found {len(breed_dirs)} breed directories")

    # Build a map of all Stanford images
    stanford_images = {}
    for breed_dir in breed_dirs:
        breed_name = os.path.basename(breed_dir)
        image_files = glob.glob(os.path.join(breed_dir, "*.jpg")) + \
                     glob.glob(os.path.join(breed_dir, "*.png"))

        for image_path in image_files:
            image_name = os.path.basename(image_path)
            stanford_images[image_name] = image_path
            # Also add without extension
            name_without_ext = os.path.splitext(image_name)[0]
            stanford_images[name_without_ext] = image_path

    print(f"  Found {len(stanford_images)} total Stanford dog images")

    # Find Stanford images in annotations
    processed_frames = {}
    processed_count = 0

    for img_id, annotation in tqdm(all_annotations.items(), desc="Processing Stanford annotations"):
        # Skip if this doesn't have an emotion
        if "emotions" not in annotation or "primary_emotion" not in annotation["emotions"]:
            continue

        # Check if this is likely a Stanford image
        is_stanford = False
        filename = os.path.basename(img_id)
        basename = os.path.splitext(filename)[0]

        # Stanford images often have format like n02085620_10074.jpg
        if re.match(r'n\d+_\d+', basename) or filename in stanford_images or basename in stanford_images:
            is_stanford = True

        if not is_stanford:
            continue

        # Find the image path
        image_path = None
        if filename in stanford_images:
            image_path = stanford_images[filename]
        elif basename in stanford_images:
            image_path = stanford_images[basename]

        if image_path:
            # Get emotion and create safe version
            emotion = annotation["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            # Create new filename
            new_filename = f"stanford_{basename}{os.path.splitext(image_path)[1]}"
            dst_path = os.path.join(combined_frames_dir, new_filename)

            # Copy to combined directory
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            try:
                shutil.copy2(image_path, dst_path)

                # Copy to class directory
                class_dir = os.path.join(output_dir, "all_by_class", safe_emotion)
                class_path = os.path.join(class_dir, new_filename)
                os.makedirs(os.path.dirname(class_path), exist_ok=True)
                shutil.copy2(image_path, class_path)

                # Add to processed frames
                processed_frames[new_filename] = {
                    "emotions": {"primary_emotion": emotion},
                    "original_id": img_id,
                    "original_path": image_path,
                    "source": "stanford"
                }

                processed_count += 1

                if processed_count % 100 == 0:
                    print(f"  Processed {processed_count} Stanford images")

            except Exception as e:
                print(f"  Error copying {image_path}: {str(e)}")

    print(f"  Processed {processed_count} Stanford images with emotion annotations")

    # Count by emotion
    emotion_counts = defaultdict(int)
    for _, data in processed_frames.items():
        emotion = data["emotions"]["primary_emotion"]
        emotion_counts[emotion] += 1

    print("  Stanford emotion distribution:")
    for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"    {emotion}: {count}")

    return processed_count, processed_frames

def create_train_val_test_splits(all_frames, output_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """Create train/val/test splits from processed frames"""
    # Group by emotion
    frames_by_emotion = defaultdict(list)
    for filename, data in all_frames.items():
        emotion = data["emotions"]["primary_emotion"]
        safe_emotion = emotion.replace("/", "_").replace("\\", "_")
        frames_by_emotion[safe_emotion].append((filename, data))

    print("\nEmotion distribution:")
    for emotion, frames in sorted(frames_by_emotion.items(), key=lambda x: len(x[1]), reverse=True):
        print(f"  {emotion}: {len(frames)} frames")

    # Perform stratified split
    train_files = []
    val_files = []
    test_files = []

    for emotion, files in frames_by_emotion.items():
        random.shuffle(files)
        n_files = len(files)
        n_train = int(n_files * train_ratio)
        n_val = int(n_files * val_ratio)

        train_files.extend(files[:n_train])
        val_files.extend(files[n_train:n_train+n_val])
        test_files.extend(files[n_train+n_val:])

    # Copy files to split directories
    split_counts = defaultdict(lambda: defaultdict(int))
    combined_frames_dir = os.path.join(output_dir, "all_frames")

    for file_list, split_dir, split_name in [
        (train_files, os.path.join(output_dir, "train_by_class"), "train"),
        (val_files, os.path.join(output_dir, "val_by_class"), "validation"),
        (test_files, os.path.join(output_dir, "test_by_class"), "test")
    ]:
        print(f"\nCreating {split_name} split with {len(file_list)} files...")

        for filename, data in tqdm(file_list, desc=f"Processing {split_name} split"):
            emotion = data["emotions"]["primary_emotion"]
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")

            src_path = os.path.join(combined_frames_dir, filename)
            dst_path = os.path.join(split_dir, safe_emotion, filename)

            if os.path.exists(src_path):
                os.makedirs(os.path.dirname(dst_path), exist_ok=True)
                shutil.copy2(src_path, dst_path)
                split_counts[split_name][safe_emotion] += 1

    # Print split statistics
    print("\nFinal Dataset Statistics:")
    for split_name in ["train", "validation", "test"]:
        emotion_counts = split_counts[split_name]
        total = sum(emotion_counts.values())

        print(f"\n{split_name.capitalize()} split:")
        print(f"  Total: {total} images")
        print("  Emotion distribution:")
        for emotion, count in sorted(emotion_counts.items(), key=lambda x: x[1], reverse=True):
            percent = count / total * 100 if total > 0 else 0
            print(f"    {emotion}: {count} ({percent:.1f}%)")
            
    # Save split counts to file for future reference
    split_stats = {
        "train": dict(split_counts["train"]),
        "validation": dict(split_counts["validation"]),
        "test": dict(split_counts["test"])
    }
    
    split_stats_path = os.path.join(output_dir, "split_statistics.json")
    with open(split_stats_path, 'w') as f:
        json.dump(split_stats, f, indent=2)
        
    print(f"\nSplit statistics saved to: {split_stats_path}")
    
    return split_counts

def process_data():
    """Run the complete data processing pipeline"""
    print("Starting Pawnder data processing pipeline...")
    
    # Set random seed for reproducibility
    random.seed(42)

    # Define key paths
    base_dir = project_root
    videos_dir = os.path.join(DATA_DIRS['personal_dataset'], "videos")
    personal_images_dir = os.path.join(DATA_DIRS['personal_dataset'], "images")
    stanford_dir = DATA_DIRS['stanford_original']
    stanford_images_dir = os.path.join(stanford_dir, "Images")
    emotions_file = os.path.join(DATA_DIRS['interim'], "emotions_only.json")

    # Output directories
    output_dir = DATA_DIRS['processed']
    combined_frames_dir = os.path.join(output_dir, "all_frames")
    os.makedirs(combined_frames_dir, exist_ok=True)

    # Create output directories
    for split in ["all_by_class", "train_by_class", "val_by_class", "test_by_class"]:
        split_dir = os.path.join(output_dir, split)
        os.makedirs(split_dir, exist_ok=True)
        for safe_name in SAFE_CLASS_NAMES:
            os.makedirs(os.path.join(split_dir, safe_name), exist_ok=True)
        os.makedirs(os.path.join(split_dir, "unknown"), exist_ok=True)



In [ ]:
# Step 1: Process video folders
    print("Step 1: Processing video folders...")
    video_folders = []
    if os.path.exists(videos_dir):
        for item in os.listdir(videos_dir):
            folder_path = os.path.join(videos_dir, item)
            if os.path.isdir(folder_path):
                video_folders.append(folder_path)

        print(f"Found {len(video_folders)} video folders")

        video_frames = {}
        total_video_frames = 0

        for video_folder in tqdm(video_folders, desc="Processing video folders"):
            count, frames = process_video_folder(video_folder, combined_frames_dir, output_dir)
            total_video_frames += count
            video_frames.update(frames)

        print(f"\nProcessed {total_video_frames} video frames from {len(video_folders)} folders")
    else:
        print(f"Video directory not found: {videos_dir}")
        video_frames = {}
        total_video_frames = 0


In [ ]:
  # Step 2: Process Stanford dataset
    print("\nStep 2: Processing Stanford dataset...")
    stanford_count, stanford_frames = process_stanford_dataset(
        stanford_images_dir, emotions_file, combined_frames_dir, output_dir
    )

In [ ]:
  # Step 3: Process personal images
    print("\nStep 3: Processing personal dataset images...")
    personal_count, personal_frames = process_personal_images(
        personal_images_dir, emotions_file, combined_frames_dir, output_dir
    )

In [ ]:
 # Step 4: Combine all annotations
    all_frames = {}
    all_frames.update(video_frames)
    all_frames.update(stanford_frames)
    all_frames.update(personal_frames)

    total_frames = len(all_frames)
    print(f"\nTotal processed frames: {total_frames}")
    print(f"  - {len(video_frames)} video frames")
    print(f"  - {len(stanford_frames)} Stanford images")
    print(f"  - {len(personal_frames)} personal images")

 # Save combined annotations
    annotations_file = os.path.join(output_dir, "combined_annotations.json")
    with open(annotations_file, 'w') as f:
        json.dump(all_frames, f, indent=2)

    print(f"Saved annotations to: {annotations_file}")


In [ ]:
 # Step 5: Create train/val/test splits
    print("\nStep 5: Creating train/val/test splits...")
    split_counts = create_train_val_test_splits(
        all_frames, 
        output_dir,
        train_ratio=config['data']['train_split'],
        val_ratio=config['data']['val_split'],
        test_ratio=config['data']['test_split']
    )
    
    print("\nDataset preparation complete!")
    return all_frames, split_counts

# Part 3: Model Architecture

In [ ]:
class DogEmotionModel:
    """Class for building dog emotion recognition models"""

    def __init__(self, config_path="config.yaml"):
        """
        Initialize the model builder

        Args:
            config_path (str): Path to configuration YAML file
        """
        self.config = load_config(config_path)

    def _create_image_branch(self, input_shape, backbone="mobilenetv2"):
        """
        Create the image processing branch of the model

        Args:
            input_shape (tuple): Image input shape (height, width, channels)
            backbone (str): Backbone model for transfer learning

        Returns:
            tuple: (input_tensor, output_tensor)
        """
        input_tensor = Input(shape=input_shape, name="image_input")

        # Select backbone based on configuration
        if backbone.lower() == "mobilenetv2":
            base_model = MobileNetV2(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "resnet50":
            base_model = ResNet50(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "efficientnetb0":
            base_model = EfficientNetB0(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # Freeze early layers for transfer learning
        for layer in base_model.layers[:int(len(base_model.layers) * 0.7)]:
            layer.trainable = False

        # Connect input to backbone
        x = base_model(input_tensor)

        # Add custom layers on top of backbone
        x = GlobalAveragePooling2D()(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"])(x)
        x = Dense(512, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"] / 2)(x)

        return input_tensor, x

    def _create_behavior_branch(self, behavior_size):
        """
        Create the behavioral indicator branch of the model

        Args:
            behavior_size (int): Number of behavioral indicator features

        Returns:
            tuple: (input_tensor, output_tensor)
        """
        input_tensor = Input(shape=(behavior_size,), name="behavior_input")

        x = Dense(128, activation="relu")(input_tensor)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"] / 2)(x)
        x = Dense(64, activation="relu")(x)
        x = BatchNormalization()(x)

        return input_tensor, x

    def _create_video_branch(self, input_shape, backbone="mobilenetv2", sequence_length=16):
        """
        Create the video processing branch of the model

        Args:
            input_shape (tuple): Frame input shape (height, width, channels)
            backbone (str): Backbone model for transfer learning
            sequence_length (int): Number of frames in the sequence

        Returns:
            tuple: (input_tensor, output_tensor)
        """
        # Input shape is (sequence_length, height, width, channels)
        input_tensor = Input(shape=(sequence_length, *input_shape), name="video_input")

        # Select backbone based on configuration
        if backbone.lower() == "mobilenetv2":
            base_model = MobileNetV2(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "resnet50":
            base_model = ResNet50(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "efficientnetb0":
            base_model = EfficientNetB0(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # Freeze early layers for transfer learning
        for layer in base_model.layers[:int(len(base_model.layers) * 0.7)]:
            layer.trainable = False

        # Apply CNN to each frame in the sequence
        frame_features = TimeDistributed(base_model)(input_tensor)
        frame_features = TimeDistributed(GlobalAveragePooling2D())(frame_features)

        # Process the sequence of frame features with LSTM
        x = Bidirectional(LSTM(256, return_sequences=True))(frame_features)
        x = Bidirectional(LSTM(128))(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"])(x)
        x = Dense(512, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"] / 2)(x)

        return input_tensor, x

    def _calculate_behavior_size(self):
        """
        Calculate the number of behavioral indicator features based on Excel data

        Returns:
            int: Number of behavioral features
        """
        # Try to load from a file
        behavior_map_path = os.path.join(
            DATA_DIRS['matrix'],
            "primary_behavior_matrix.json"
        )

        if os.path.exists(behavior_map_path):
            with open(behavior_map_path, 'r') as f:
                behavior_map = json.load(f)
                
            # Count total behaviors across all categories
            behavior_count = 0
            for category in behavior_map.get('behavior_categories', []):
                behavior_count += len(category.get('behaviors', []))
                
            if behavior_count > 0:
                return behavior_count

        # Default size if file not found or is empty
        return 64  # Assume 64 behavioral indicators by default

    def _create_emotion_head(self, x, num_emotions=7):
        """
        Create the emotion classification head

        Args:
            x: Input tensor
            num_emotions (int): Number of emotion classes

        Returns:
            tf.Tensor: Output tensor
        """
        emotion_output = Dense(num_emotions, activation="softmax", name="emotion_output")(x)
        return emotion_output

    def _create_confidence_head(self, x):
        """
        Create the confidence scoring head

        Args:
            x: Input tensor

        Returns:
            tf.Tensor: Output tensor
        """
        confidence_output = Dense(1, activation="sigmoid", name="confidence_output")(x)
        return confidence_output

    def create_model(self, model_type="image", num_emotions=None, behavior_size=None):
        """
        Create the full model architecture

        Args:
            model_type (str): 'image' or 'video'
            num_emotions (int): Number of emotion classes
            behavior_size (int): Number of behavioral indicator features

        Returns:
            tf.keras.Model: Compiled model
        """
        # Set up image size from config
        input_shape = tuple(self.config["model"]["image_size"])
        
        # Default number of emotions if not provided
        if num_emotions is None:
            num_emotions = len(CLASS_NAMES)

        # Calculate behavior size if not provided
        if behavior_size is None:
            behavior_size = self._calculate_behavior_size()

        # Get backbone type from config
        backbone = self.config["model"]["backbone"]

        # Create model branches
        if model_type.lower() == "image":
            image_input, image_features = self._create_image_branch(input_shape, backbone)
            behavior_input, behavior_features = self._create_behavior_branch(behavior_size)

            # Combine features
            combined = Concatenate()([image_features, behavior_features])

        elif model_type.lower() == "video":
            sequence_length = self.config["model"].get("sequence_length", 16)
            video_input, video_features = self._create_video_branch(input_shape, backbone, sequence_length)
            behavior_input, behavior_features = self._create_behavior_branch(behavior_size)

            # Combine features
            combined = Concatenate()([video_features, behavior_features])

        else:
            raise ValueError(f"Unsupported model type: {model_type}")

        # Add fusion layers
        x = Dense(256, activation="relu")(combined)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"] / 4)(x)

        # Create output heads
        emotion_output = self._create_emotion_head(x, num_emotions)
        confidence_output = self._create_confidence_head(x)

        # Create model with appropriate inputs
        if model_type.lower() == "image":
            model = Model(
                inputs=[image_input, behavior_input],
                outputs=[emotion_output, confidence_output]
            )
        else:  # video
            model = Model(
                inputs=[video_input, behavior_input],
                outputs=[emotion_output, confidence_output]
            )

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=self.config["model"]["learning_rate"]),
            loss={
                "emotion_output": "categorical_crossentropy",
                "confidence_output": "binary_crossentropy"
            },
            metrics={
                "emotion_output": ["accuracy", "top_k_categorical_accuracy"],
                "confidence_output": ["accuracy"]
            },
            loss_weights={
                "emotion_output": 1.0,
                "confidence_output": 0.2
            }
        )

        return model

    def create_feature_extractor(self):
        """
        Create a feature extractor model to extract dog visual features

        Returns:
            tf.keras.Model: Feature extractor model
        """
        # Set up image size from config
        input_shape = tuple(self.config["model"]["image_size"])

        # Get backbone type from config
        backbone = self.config["model"]["backbone"]

        # Create image branch
        image_input, image_features = self._create_image_branch(input_shape, backbone)

        # Add a few more layers
        x = Dense(256, activation="relu")(image_features)
        x = BatchNormalization()(x)
        features = Dropout(self.config["model"]["dropout_rate"] / 2)(x)

        # Create model
        model = Model(inputs=image_input, outputs=features)

        return model

    def create_region_attention_model(self, num_emotions=None, behavior_size=None, num_regions=5):
        """
        Create a model with attention to different regions of dog (eyes, ears, mouth, tail, body)

        Args:
            num_emotions (int): Number of emotion classes
            behavior_size (int): Number of behavioral indicator features
            num_regions (int): Number of body regions to model

        Returns:
            tf.keras.Model: Compiled model with region attention
        """
        # Default number of emotions if not provided
        if num_emotions is None:
            num_emotions = len(CLASS_NAMES)
            
        # Set up image size from config
        input_shape = tuple(self.config["model"]["image_size"])

        # Calculate behavior size if not provided
        if behavior_size is None:
            behavior_size = self._calculate_behavior_size()

        # Get backbone type from config
        backbone = self.config["model"]["backbone"]

        # Main image input
        image_input = Input(shape=input_shape, name="image_input")

        # Region coordinates inputs (each region has [x, y, width, height])
        region_inputs = []
        region_features = []

        # Create a mini-model for the backbone
        if backbone.lower() == "mobilenetv2":
            base_model = MobileNetV2(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "resnet50":
            base_model = ResNet50(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        elif backbone.lower() == "efficientnetb0":
            base_model = EfficientNetB0(
                input_shape=input_shape,
                include_top=False,
                weights="imagenet"
            )
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        # Freeze early layers for transfer learning
        for layer in base_model.layers[:int(len(base_model.layers) * 0.7)]:
            layer.trainable = False

        # Process full image
        full_features = base_model(image_input)
        full_features = GlobalAveragePooling2D()(full_features)

        # Process each region
        for i in range(num_regions):
            # Input for region coordinates [x, y, width, height]
            region_input = Input(shape=(4,), name=f"region_{i}_input")
            region_inputs.append(region_input)

            # Extract region features using a Lambda layer with custom cropping function
            from tensorflow.keras.layers import Lambda

            def extract_region(inputs):
                img, coords = inputs
                # Get coordinates
                x, y, w, h = coords[0]

                # Convert to integers
                x = tf.cast(x * tf.cast(tf.shape(img)[2], dtype=tf.float32), dtype=tf.int32)
                y = tf.cast(y * tf.cast(tf.shape(img)[1], dtype=tf.float32), dtype=tf.int32)
                w = tf.cast(w * tf.cast(tf.shape(img)[2], dtype=tf.float32), dtype=tf.int32)
                h = tf.cast(h * tf.cast(tf.shape(img)[1], dtype=tf.float32), dtype=tf.int32)

                # Ensure coordinates are valid
                x = tf.maximum(0, x)
                y = tf.maximum(0, y)
                w = tf.minimum(w, tf.shape(img)[2] - x)
                h = tf.minimum(h, tf.shape(img)[1] - y)

                # Extract region
                region = tf.image.crop_to_bounding_box(img, y, x, h, w)

                # Resize to standard size
                region = tf.image.resize(region, [input_shape[0], input_shape[1]])

                return region

            # Extract and process region
            region = Lambda(extract_region)([image_input, region_input])
            region_feature = base_model(region)
            region_feature = GlobalAveragePooling2D()(region_feature)
            region_features.append(region_feature)

        # Behavioral input
        behavior_input, behavior_features = self._create_behavior_branch(behavior_size)

        # Combine features from full image, regions, and behavior
        all_features = [full_features] + region_features + [behavior_features]
        combined = Concatenate()(all_features)

        # Add fusion layers
        x = Dense(512, activation="relu")(combined)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"])(x)
        x = Dense(256, activation="relu")(x)
        x = BatchNormalization()(x)
        x = Dropout(self.config["model"]["dropout_rate"] / 2)(x)

        # Create output heads
        emotion_output = self._create_emotion_head(x, num_emotions)
        confidence_output = self._create_confidence_head(x)

        # Create full model
        model = Model(
            inputs=[image_input] + region_inputs + [behavior_input],
            outputs=[emotion_output, confidence_output]
        )

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=self.config["model"]["learning_rate"]),
            loss={
                "emotion_output": "categorical_crossentropy",
                "confidence_output": "binary_crossentropy"
            },
            metrics={
                "emotion_output": ["accuracy", "top_k_categorical_accuracy"],
                "confidence_output": ["accuracy"]
            },
            loss_weights={
                "emotion_output": 1.0,
                "confidence_output": 0.2
            }
        )

        return model

    def create_model_with_uncertainty(self, model_type="image", num_emotions=None, behavior_size=None):
        """
        Create a model with uncertainty estimation using Monte Carlo Dropout

        Args:
            model_type (str): 'image' or 'video'
            num_emotions (int): Number of emotion classes
            behavior_size (int): Number of behavioral indicator features

        Returns:
            tf.keras.Model: Compiled model with uncertainty estimation
        """
        # Default number of emotions if not provided
        if num_emotions is None:
            num_emotions = len(CLASS_NAMES)
            
        # Create base model
        model = self.create_model(model_type, num_emotions, behavior_size)

        # Create a wrapper function for Monte Carlo Dropout inference
        def mc_dropout_predict(inputs, n_samples=10):
            """Perform Monte Carlo Dropout inference"""
            # Enable dropout at inference time
            tf.keras.backend.set_learning_phase(1)

            # Initialize arrays for predictions
            emotion_preds = []
            confidence_preds = []

            # Perform multiple forward passes
            for _ in range(n_samples):
                emotion_pred, confidence_pred = model.predict(inputs)
                emotion_preds.append(emotion_pred)
                confidence_preds.append(confidence_pred)

            # Stack predictions
            emotion_preds = np.stack(emotion_preds, axis=0)
            confidence_preds = np.stack(confidence_preds, axis=0)

            # Calculate mean and standard deviation
            emotion_mean = np.mean(emotion_preds, axis=0)
            emotion_std = np.std(emotion_preds, axis=0)
            confidence_mean = np.mean(confidence_preds, axis=0)
            confidence_std = np.std(confidence_preds, axis=0)

            # Disable dropout for normal inference
            tf.keras.backend.set_learning_phase(0)

            return (emotion_mean, emotion_std), (confidence_mean, confidence_std)

        # Attach the function to the model for easy access
        model.mc_dropout_predict = mc_dropout_predict

        return model


# Part 4: Data Generator and Training

In [ ]:
class DogEmotionDataGenerator(tf.keras.utils.Sequence):
    """Custom data generator for dog emotion recognition model"""
    
    def __init__(self, data_dir, split='train', img_size=(224, 224), batch_size=32, 
                 shuffle=True, augment=False, behavior_size=64, use_behaviors=True):
        """
        Initialize the data generator
        
        Args:
            data_dir (str): Base directory containing data
            split (str): 'train', 'val', or 'test'
            img_size (tuple): Target image size (height, width)
            batch_size (int): Batch size
            shuffle (bool): Whether to shuffle data between epochs
            augment (bool): Whether to apply data augmentation
            behavior_size (int): Size of behavior feature vector
            use_behaviors (bool): Whether to use behavior features
        """
        self.data_dir = data_dir
        self.split = split
        self.img_size = img_size
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        self.behavior_size = behavior_size
        self.use_behaviors = use_behaviors
        
        # Set proper split directory name
        if split == 'val':
            split_dir_name = 'val_by_class'
        elif split == 'test':
            split_dir_name = 'test_by_class'
        else:
            split_dir_name = 'train_by_class'
            
        self.split_dir = os.path.join(data_dir, split_dir_name)
        
        # Load image paths and labels
        self.samples = []
        self.emotion_to_idx = {}
        
        # Map emotions to indices
        for idx, emotion in enumerate(sorted(CLASS_NAMES)):
            safe_emotion = emotion.replace("/", "_").replace("\\", "_")
            self.emotion_to_idx[emotion] = idx
            
            # Find all images for this emotion
            emotion_dir = os.path.join(self.split_dir, safe_emotion)
            if os.path.exists(emotion_dir):
                for img_file in os.listdir(emotion_dir):
                    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(emotion_dir, img_file)
                        self.samples.append((img_path, emotion))
        
        # Shuffle if needed
        if self.shuffle:
            random.shuffle(self.samples)
            
        print(f"Created {split} generator with {len(self.samples)} samples")
        
        # Set up augmentation if needed
        if self.augment:
            self.img_gen = ImageDataGenerator(
                rotation_range=20,
                width_shift_range=0.2,
                height_shift_range=0.2,
                zoom_range=0.2,
                horizontal_flip=True,
                fill_mode='nearest'
            )
            
        # Try to load behavioral matrix
        self.behavior_matrix = self._load_behavior_matrix()
    
    def _load_behavior_matrix(self):
        """Load behavioral matrix from JSON file if available"""
        behavior_map_path = os.path.join(DATA_DIRS['matrix'], "primary_behavior_matrix.json")
        if os.path.exists(behavior_map_path):
            try:
                with open(behavior_map_path, 'r') as f:
                    return json.load(f)
            except:
                pass
        return None
    
    def __len__(self):
        """Return the number of batches per epoch"""
        return int(np.ceil(len(self.samples) / self.batch_size))
    
    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min((idx + 1) * self.batch_size, len(self.samples))
        batch_samples = self.samples[start_idx:end_idx]
        batch_size = len(batch_samples)
        
        # Initialize batch arrays
        batch_images = np.zeros((batch_size, self.img_size[0], self.img_size[1], 3), dtype=np.float32)
        batch_emotions = np.zeros((batch_size, len(self.emotion_to_idx)), dtype=np.float32)
        batch_behaviors = np.zeros((batch_size, self.behavior_size), dtype=np.float32)
        batch_confidence = np.ones((batch_size, 1), dtype=np.float32)  # Default confidence of 1.0
        
        # Fill batch data
        for i, (img_path, emotion) in enumerate(batch_samples):
            try:
                # Load and preprocess image
                img = self._load_and_preprocess_image(img_path)
                batch_images[i] = img
                
                # One-hot encode emotion
                emotion_idx = self.emotion_to_idx.get(emotion, 0)
                batch_emotions[i, emotion_idx] = 1.0
                
                # Generate behavioral features (placeholder for now)
                # In a real implementation, you would extract these from annotations or matrix
                if self.use_behaviors and self.behavior_matrix:
                    # Try to extract some behavioral features based on emotion
                    behavior_states = self.behavior_matrix.get('behavioral_states', [])
                    state_id = None
                    
                    # Find the state ID matching the emotion
                    for state in behavior_states:
                        if state['name'] == emotion:
                            state_id = state['id']
                            break
                    
                    # Generate behavioral features from the matrix if we found the state
                    if state_id:
                        behavior_idx = 0
                        for category in self.behavior_matrix.get('behavior_categories', []):
                            for behavior in category.get('behaviors', []):
                                if behavior_idx < self.behavior_size:
                                    mapping = behavior.get('state_mapping', {})
                                    if state_id in mapping:
                                        batch_behaviors[i, behavior_idx] = mapping[state_id]
                                    behavior_idx += 1
                
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                # Lower confidence if there was an error
                batch_confidence[i] = 0.5
        
        # Return inputs and outputs
        inputs = {
            'image_input': batch_images,
            'behavior_input': batch_behaviors
        }
        outputs = {
            'emotion_output': batch_emotions,
            'confidence_output': batch_confidence
        }
        
        return inputs, outputs
    
    def on_epoch_end(self):
        """Shuffle indices at the end of each epoch if needed"""
        if self.shuffle:
            random.shuffle(self.samples)
    
    def _load_and_preprocess_image(self, img_path):
        """Load and preprocess an image"""
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError(f"Could not load image: {img_path}")
        
        # Convert BGR to RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Resize to target size
        img = cv2.resize(img, (self.img_size[1], self.img_size[0]))
        
        # Normalize pixel values to [0, 1]
        img = img.astype(np.float32) / 255.0
        
        # Apply augmentation if needed
        if self.augment:
            img = self.img_gen.random_transform(img)
        
        return img

def train_model(model, data_dir, config, model_name="dog_emotion"):
    """
    Train the model
    
    Args:
        model (tf.keras.Model): Model to train
        data_dir (str): Directory containing the data
        config (dict): Configuration dictionary
        model_name (str): Name for saved model
        
    Returns:
        tuple: (trained_model, history)
    """
    # Set up parameters
    batch_size = config['model']['batch_size']
    img_size = tuple(config['model']['image_size'][:2])  # Height, width only
    epochs = config['training'].get('epochs', 50)
    
    # Create data generators
    train_gen = DogEmotionDataGenerator(
        data_dir=data_dir,
        split='train',
        img_size=img_size,
        batch_size=batch_size,
        shuffle=True,
        augment=True
    )
    
    val_gen = DogEmotionDataGenerator(
        data_dir=data_dir,
        split='val',
        img_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        augment=False
    )
    
    # Create checkpoint directory
    checkpoint_dir = config['training']['checkpoint_dir']
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    logs_dir = config['training']['logs_dir']
    os.makedirs(logs_dir, exist_ok=True)
    
    # Create timestamp for unique folder names
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    model_dir = os.path.join(checkpoint_dir, f"{model_name}_{timestamp}")
    log_dir = os.path.join(logs_dir, f"{model_name}_{timestamp}")
    
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)
    
    # Define callbacks
    callbacks = [
        # Early stopping
        EarlyStopping(
            monitor='val_emotion_output_accuracy',
            patience=config['model']['early_stopping_patience'],
            mode='max',
            verbose=1
        ),
        
        # Reduce learning rate on plateau
        ReduceLROnPlateau(
            monitor='val_emotion_output_accuracy',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            mode='max',
            verbose=1
        ),
        
        # Save best model
        ModelCheckpoint(
            os.path.join(model_dir, 'best_model.h5'),
            monitor='val_emotion_output_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        
        # Save regular checkpoints
        ModelCheckpoint(
            os.path.join(model_dir, 'model_epoch_{epoch:02d}.h5'),
            save_freq='epoch',
            verbose=0
        ),
        
        # TensorBoard logs
        TensorBoard(
            log_dir=log_dir,
            histogram_freq=1,
            write_graph=True,
            update_freq='epoch'
        )
    ]
    
    # Train the model
    print(f"Starting training for {model_name}...")
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    # Save the final model
    final_model_path = os.path.join(model_dir, f"{model_name}_final.h5")
    model.save(final_model_path)
    print(f"Final model saved to {final_model_path}")
    
    # Save training history
    history_path = os.path.join(model_dir, f"{model_name}_history.json")
    
    with open(history_path, 'w') as f:
        # Convert numpy arrays to lists for JSON serialization
        history_dict = {}
        for key, values in history.history.items():
            history_dict[key] = [float(val) for val in values]
        
        json.dump(history_dict, f, indent=2)
    
    print(f"Training history saved to {history_path}")
    
    # Plot training history
    plot_training_history(history, model_name, model_dir)
    
    return model, history

def plot_training_history(history, model_name, save_dir):
    """Plot and save training history"""
    # Create the figure
    plt.figure(figsize=(12, 5))
    
    # Plot accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['emotion_output_accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_emotion_output_accuracy'], label='Validation Accuracy')
    plt.title('Emotion Recognition Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # Plot loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['emotion_output_loss'], label='Training Loss')
    plt.plot(history.history['val_emotion_output_loss'], label='Validation Loss')
    plt.title('Emotion Recognition Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.tight_layout()
    
    # Save plot
    history_plot_path = os.path.join(save_dir, f"{model_name}_training_history.png")
    plt.savefig(history_plot_path, dpi=300)
    plt.close()
    
    print(f"Training history plot saved to {history_plot_path}")

def evaluate_model(model, data_dir, config, model_name="dog_emotion"):
    """
    Evaluate the trained model on the test set
    
    Args:
        model (tf.keras.Model): Trained model
        data_dir (str): Directory containing the data
        config (dict): Configuration dictionary
        model_name (str): Name for saving evaluation results
        
    Returns:
        dict: Evaluation metrics
    """
    # Set up parameters
    batch_size = config['model']['batch_size']
    img_size = tuple(config['model']['image_size'][:2])  # Height, width only
    
    # Create test generator
    test_gen = DogEmotionDataGenerator(
        data_dir=data_dir,
        split='test',
        img_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        augment=False
    )
    
    # Evaluate the model
    print(f"Evaluating {model_name} on test set...")
    evaluation = model.evaluate(test_gen, verbose=1)
    
    # Get evaluation metrics
    metric_names = model.metrics_names
    evaluation_dict = {name: float(value) for name, value in zip(metric_names, evaluation)}
    
    # Print evaluation results
    print("\nEvaluation Results:")
    for name, value in evaluation_dict.items():
        print(f"{name}: {value:.4f}")
    
    # Save evaluation results
    checkpoint_dir = config['training']['checkpoint_dir']
    eval_path = os.path.join(checkpoint_dir, f"{model_name}_evaluation.json")
    
    with open(eval_path, 'w') as f:
        json.dump(evaluation_dict, f, indent=2)
    
    print(f"Evaluation results saved to {eval_path}")
    
    # Generate confusion matrix and classification report
    generate_classification_metrics(model, test_gen, model_name, checkpoint_dir)
    
    return evaluation_dict

def generate_classification_metrics(model, test_gen, model_name, save_dir):
    """
    Generate and save confusion matrix and classification report
    
    Args:
        model (tf.keras.Model): Trained model
        test_gen (DogEmotionDataGenerator): Test data generator
        model_name (str): Name for saving results
        save_dir (str): Directory to save results
    """
    # Get predictions
    y_true = []
    y_pred = []
    
    print("Generating predictions for classification metrics...")
    for i in tqdm(range(len(test_gen))):
        inputs, outputs = test_gen[i]
        batch_pred = model.predict(inputs, verbose=0)
        
        # Extract true and predicted emotions
        batch_true = np.argmax(outputs['emotion_output'], axis=1)
        batch_pred = np.argmax(batch_pred[0], axis=1)  # First output is emotion probabilities
        
        y_true.extend(batch_true)
        y_pred.extend(batch_pred)
    
    # Convert to numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Get class names
    class_names = list(test_gen.emotion_to_idx.keys())
    
    # Generate confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm_norm, 
        annot=True, 
        fmt='.2f', 
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title(f'Normalized Confusion Matrix - {model_name}')
    plt.ylabel('True Emotion')
    plt.xlabel('Predicted Emotion')
    
    # Save confusion matrix plot
    cm_path = os.path.join(save_dir, f"{model_name}_confusion_matrix.png")
    plt.savefig(cm_path, bbox_inches='tight', dpi=300)
    plt.close()
    
    # Generate classification report
    report = classification_report(
        y_true, 
        y_pred, 
        target_names=class_names,
        output_dict=True
    )
    
    # Save classification report
    report_path = os.path.join(save_dir, f"{model_name}_classification_report.json")
    
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    
    print(f"Confusion matrix saved to {cm_path}")
    print(f"Classification report saved to {report_path}")


# Part 5: Inference and Visualization

In [ ]:
def predict_image(model, image_path, config, class_names=None, show_visualization=True):
    """
    Predict emotion from a single image
    
    Args:
        model (tf.keras.Model): Trained model
        image_path (str): Path to the image
        config (dict): Configuration dictionary
        class_names (list): List of emotion class names
        show_visualization (bool): Whether to display visualization
        
    Returns:
        dict: Prediction results
    """
    if class_names is None:
        class_names = CLASS_NAMES
        
    img_size = tuple(config['model']['image_size'][:2])
    
    # Load and preprocess image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image: {image_path}")
    
    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Preprocess image
    img_resized = cv2.resize(img_rgb, (img_size[1], img_size[0]))
    img_norm = img_resized.astype(np.float32) / 255.0
    
    # Create dummy behavior input (all zeros for now)
    behavior_size = model.inputs[1].shape[1]
    behavior_input = np.zeros((1, behavior_size), dtype=np.float32)
    
    # Make prediction
    inputs = {
        'image_input': np.expand_dims(img_norm, axis=0),
        'behavior_input': behavior_input
    }
    
    predictions = model.predict(inputs, verbose=0)
    emotion_probs, confidence = predictions
    
    # Get the predicted emotion
    emotion_idx = np.argmax(emotion_probs[0])
    emotion = class_names[emotion_idx]
    emotion_score = float(emotion_probs[0][emotion_idx])
    confidence_score = float(confidence[0][0])
    
    # Create result dictionary
    result = {
        'emotion': emotion,
        'emotion_score': emotion_score,
        'confidence': confidence_score,
        'all_emotions': {class_names[i]: float(emotion_probs[0][i]) for i in range(len(class_names))}
    }
    
    # Visualize result if requested
    if show_visualization:
        visualize_prediction(img_rgb, result, image_path)
    
    return result

def visualize_prediction(img, result, image_path, save_path=None):
    """
    Visualize prediction result
    
    Args:
        img (np.ndarray): Image array in RGB format
        result (dict): Prediction result
        image_path (str): Path to the original image
        save_path (str): Path to save visualization (optional)
    """
    # Create figure
    plt.figure(figsize=(12, 6))
    
    # Display image
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f"Predicted: {result['emotion']} ({result['emotion_score']:.2f})")
    plt.axis('off')
    
    # Plot emotion probabilities
    plt.subplot(1, 2, 2)
    emotions = list(result['all_emotions'].keys())
    scores = list(result['all_emotions'].values())
    
    # Sort by score
    sorted_indices = np.argsort(scores)[::-1]
    emotions = [emotions[i] for i in sorted_indices]
    scores = [scores[i] for i in sorted_indices]
    
    # Bar chart for top 5 emotions
    bars = plt.barh(emotions[:5], scores[:5], color='skyblue')
    
    # Add confidence score
    plt.axvline(x=result['confidence'], color='red', linestyle='--', label=f'Confidence: {result["confidence"]:.2f}')
    
    # Highlight predicted emotion
    for i, e in enumerate(emotions[:5]):
        if e == result['emotion']:
            bars[i].set_color('green')
    
    plt.xlabel('Score')
    plt.ylabel('Emotion')
    plt.title('Top 5 Emotion Predictions')
    plt.xlim(0, 1)
    plt.legend()
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    
    plt.show()

def process_video(model, video_path, config, class_names=None, output_path=None, frame_interval=5):
    """
    Process a video and predict emotions
    
    Args:
        model (tf.keras.Model): Trained model
        video_path (str): Path to the video
        config (dict): Configuration dictionary
        class_names (list): List of emotion class names
        output_path (str): Path to save output video (optional)
        frame_interval (int): Process every Nth frame
        
    Returns:
        dict: Prediction results over time
    """
    if class_names is None:
        class_names = CLASS_NAMES
        
    img_size = tuple(config['model']['image_size'][:2])
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Setup output video if requested
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(
            output_path,
            fourcc,
            fps / frame_interval,  # Adjust output FPS based on frame interval
            (frame_width, frame_height)
        )
    
    # Create dummy behavior input (all zeros for now)
    behavior_size = model.inputs[1].shape[1]
    behavior_input = np.zeros((1, behavior_size), dtype=np.float32)
    
    # Process video frames
    results = {
        'video_path': video_path,
        'fps': fps,
        'total_frames': total_frames,
        'frames_analyzed': 0,
        'emotion_timeline': [],
        'dominant_emotion': None
    }
    
    frame_count = 0
    emotion_counts = {emotion: 0 for emotion in class_names}
    
    print(f"Processing video: {video_path}")
    
    try:
        with tqdm(total=total_frames) as pbar:
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                
                # Process every Nth frame
                if frame_count % frame_interval == 0:
                    # Convert to RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    
                    # Preprocess image
                    img_resized = cv2.resize(frame_rgb, (img_size[1], img_size[0]))
                    img_norm = img_resized.astype(np.float32) / 255.0
                    
                    # Make prediction
                    inputs = {
                        'image_input': np.expand_dims(img_norm, axis=0),
                        'behavior_input': behavior_input
                    }
                    
                    predictions = model.predict(inputs, verbose=0)
                    emotion_probs, confidence = predictions
                    
                    # Get the predicted emotion
                    emotion_idx = np.argmax(emotion_probs[0])
                    emotion = class_names[emotion_idx]
                    emotion_score = float(emotion_probs[0][emotion_idx])
                    confidence_score = float(confidence[0][0])
                    
                    # Count emotions for dominant emotion calculation
                    emotion_counts[emotion] += 1
                    
                    # Add to timeline
                    results['emotion_timeline'].append({
                        'frame': frame_count,
                        'time': frame_count / fps,
                        'emotion': emotion,
                        'emotion_score': emotion_score,
                        'confidence': confidence_score
                    })
                    
                    # Update count of analyzed frames
                    results['frames_analyzed'] += 1
                    
                    # Draw results on frame for output video
                    if output_path:
                        # Add text
                        text = f"{emotion}: {emotion_score:.2f}, Conf: {confidence_score:.2f}"
                        cv2.putText(
                            frame, text, (10, 30), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2
                        )
                        
                        # Write frame to output video
                        out.write(frame)
                
                frame_count += 1
                pbar.update(1)
    
    finally:
        # Release resources
        cap.release()
        if output_path and 'out' in locals():
            out.release()
    
    # Calculate dominant emotion
    if results['frames_analyzed'] > 0:
        dominant_emotion = max(emotion_counts.items(), key=lambda x: x[1])[0]
        results['dominant_emotion'] = dominant_emotion
        
        # Calculate emotion distribution
        results['emotion_distribution'] = {
            emotion: count / results['frames_analyzed'] if results['frames_analyzed'] > 0 else 0
            for emotion, count in emotion_counts.items()
        }
    
    # Visualize emotion over time
    if results['frames_analyzed'] > 0:
        visualize_video_emotions(results, class_names, output_path)
    
    return results

def visualize_video_emotions(results, class_names, output_path=None):
    """
    Visualize emotions over time in a video
    
    Args:
        results (dict): Video processing results
        class_names (list): List of emotion class names
        output_path (str): Path to save visualization (optional)
    """
    # Create figure
    plt.figure(figsize=(14, 10))
    
    # Plot emotion distribution
    plt.subplot(2, 1, 1)
    emotions = []
    percentages = []
    
    for emotion, percent in sorted(results['emotion_distribution'].items(), key=lambda x: x[1], reverse=True):
        if percent > 0:
            emotions.append(emotion)
            percentages.append(percent)
    
    bars = plt.bar(emotions, percentages, color='skyblue')
    
    # Highlight dominant emotion
    for i, emotion in enumerate(emotions):
        if emotion == results['dominant_emotion']:
            bars[i].set_color('green')
    
    plt.xlabel('Emotion')
    plt.ylabel('Percentage')
    plt.title('Emotion Distribution')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')
    
    # Plot emotion timeline
    plt.subplot(2, 1, 2)
    
    # Convert emotions to numeric values
    emotion_to_idx = {emotion: i for i, emotion in enumerate(class_names)}
    times = [entry['time'] for entry in results['emotion_timeline']]
    emotion_values = [emotion_to_idx[entry['emotion']] for entry in results['emotion_timeline']]
    
    plt.scatter(times, emotion_values, alpha=0.6)
    plt.xlabel('Time (seconds)')
    plt.ylabel('Emotion')
    plt.title('Emotion Timeline')
    plt.yticks(range(len(class_names)), class_names)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save if requested
    if output_path:
        # Create a path for the visualization by modifying the output video path
        vis_path = os.path.splitext(output_path)[0] + "_emotions.png"
        plt.savefig(vis_path, dpi=300, bbox_inches='tight')
        print(f"Emotion visualization saved to {vis_path}")
    
    plt.show()

# Part 6: Main Execution

In [ ]:
def main(args=None):
    """Main execution function"""
    import argparse
    
    # Parse command line arguments if provided
    if args is None:
        parser = argparse.ArgumentParser(description='Dog Emotion Recognition Pipeline')
        parser.add_argument('--process_data', action='store_true', help='Process the dataset')
        parser.add_argument('--train', action='store_true', help='Train the model')
        parser.add_argument('--evaluate', action='store_true', help='Evaluate the model')
        parser.add_argument('--predict', type=str, help='Predict emotion for an image file')
        parser.add_argument('--process_video', type=str, help='Process a video file')
        parser.add_argument('--model_type', type=str, default='image', choices=['image', 'video', 'region'], 
                            help='Model type to use')
        parser.add_argument('--model_path', type=str, help='Path to a saved model for prediction/evaluation')
        parser.add_argument('--output', type=str, help='Output path for video processing')
        args = parser.parse_args()
    
    # Make sure at least one action is specified
    if not any([args.process_data, args.train, args.evaluate, args.predict, args.process_video]):
        print("No action specified. Please use --process_data, --train, --evaluate, --predict, or --process_video")
        return
    
    # Process data if requested
    if args.process_data:
        all_frames, split_counts = process_data()
        print("Data processing complete!")
    
    # Train model if requested
    if args.train:
        print(f"Training a {args.model_type} model...")
        
        # Create model builder
        model_builder = DogEmotionModel()
        
        # Create model based on specified type
        if args.model_type == 'region':
            model = model_builder.create_region_attention_model()
        elif args.model_type == 'video':
            model = model_builder.create_model(model_type='video')
        else:
            model = model_builder.create_model(model_type='image')
        
        # Train the model
        model, history = train_model(
            model, 
            DATA_DIRS['processed'], 
            config, 
            model_name=f"dog_emotion_{args.model_type}"
        )
        
        # Save model path for evaluation or prediction
        args.model_path = os.path.join(
            config['training']['checkpoint_dir'],
            f"dog_emotion_{args.model_type}_final.h5"
        )
        
        print(f"Training complete! Model saved to {args.model_path}")
    
    # Evaluate model if requested
    if args.evaluate:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for evaluation. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Evaluate the model
        evaluation = evaluate_model(
            model, 
            DATA_DIRS['processed'], 
            config, 
            model_name=os.path.basename(args.model_path).split('.')[0]
        )
        
        print("Evaluation complete!")
    
    # Predict for a single image if requested
    if args.predict:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for prediction. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Make prediction
        result = predict_image(model, args.predict, config)
        
        print(f"\nPrediction for {args.predict}:")
        print(f"Emotion: {result['emotion']} (Score: {result['emotion_score']:.2f})")
        print(f"Confidence: {result['confidence']:.2f}")
        
        print("\nTop 3 emotions:")
        for emotion, score in sorted(result['all_emotions'].items(), key=lambda x: x[1], reverse=True)[:3]:
            print(f"- {emotion}: {score:.2f}")
    
    # Process video if requested
    if args.process_video:
        # Make sure we have a model
        if args.model_path is None and 'model' not in locals():
            print("No model available for video processing. Please specify --model_path or run --train first.")
            return
        
        # Load model if needed
        if 'model' not in locals():
            print(f"Loading model from {args.model_path}")
            model = tf.keras.models.load_model(args.model_path)
        
        # Determine output path
        output_path = args.output
        if output_path is None:
            # Create default output path
            base_name = os.path.basename(args.process_video)
            output_dir = os.path.join(DATA_DIRS['processed'], "predictions")
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, f"analyzed_{base_name}")
        
        # Process video
        results = process_video(model, args.process_video, config, output_path=output_path)
        
        print(f"\nVideo Analysis Results for {args.process_video}")
        print(f"Dominant Emotion: {results['dominant_emotion']}")
        print("\nEmotion Distribution:")
        for emotion, percent in sorted(
            results['emotion_distribution'].items(), 
            key=lambda x: x[1], 
            reverse=True
        ):
            print(f"  {emotion}: {percent:.1%}")
        
        print(f"\nProcessed video saved to: {output_path}")

if __name__ == "__main__":
    main()